# Entropy generator: nozzles with losses — replicating De Domenico et al. (2019)

This notebook validates the **FNS perturbation network** against

> F. De Domenico, E. O. Rolland, S. Hochgreb, *A generalised model for acoustic
> and entropic transfer function of nozzles with losses*, **J. Sound Vib. 440
> (2019) 212–230** (`scratch/De Domenico et al. - 2019 ...pdf`).

That paper is a **compact** (low-frequency) model, so the FNS `omega -> 0` jumps
are the *exact* regime — no non-compact machinery is needed.  A lossy nozzle is
four stations (their Fig. 3),

$$A_1 \;\xrightarrow{\text{isentropic}}\; A_T \;\xrightarrow{\text{isentropic}}\; A_j \;\xrightarrow{\text{Borda (momentum)}}\; A_2,$$

isentropic up to a notional area $A_j$ in the divergent, then a
momentum-conserving (Borda–Carnot) jump $A_j\!\to\!A_2$ for the loss.  The single
loss parameter is $\beta = A_j/A_2$:

* $\beta = \beta_{\min} = A_T/A_2$  → **orifice plate** (jet, maximum loss),
* $\beta = 1$  → **isentropic nozzle** (full pressure recovery, no loss),
* in between → a **non-isentropic nozzle**.

We build these from just two FNS elements — `isentropic_area_change` and
`sudden_area_change` (whose momentum law *is* their Eq. 5/6) — and check, at the
**Cambridge Entropy Generator** geometry ($D_1=42.6$ mm, $D_T=6.6$ mm):

1. **Mean flow** — the upstream static-pressure rise $p_1/p_2$ vs Mach (their Fig. 5/9).
2. **Analytical anchor** — the assembled compact scattering matrix in the De Domenico
   $(P^+,P^-,\sigma)$ flavour (our `riemann` basis, their Eqs. 9–11) against an
   *independent* composition of the analytic jumps.
3. **Transfer functions** — $R^\pm, T^\pm$ and the entropy→sound coefficients vs throat
   Mach (their Fig. 6).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fns.elements import catalog as cat
from fns.thermo.configure import perfect_gas
from fns.solver import solve
from fns.solver.control import states_table
from fns.derive import ES_RHO, ES_U, ES_P, ES_C, ES_M, ES_PT
from fns.perturbation import perturbation_response
from fns.perturbation.characteristics import char_to_dq
from fns.plotting import use_fns_theme, COLORWAY

use_fns_theme()

R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
K = CP / R
CFG = perfect_gas(R, GAMMA)
FULL = ("acoustic", "entropy")               # drive the entropy wave too -> full 3x3

# Cambridge Entropy Generator geometry (circular ducts)
D1, DT = 0.0426, 0.0066
A1 = A2 = np.pi / 4 * D1 ** 2
AT = np.pi / 4 * DT ** 2
# The sudden element is a smooth Borda<->isentropic switch; for one-directional flow
# a sharp per-element eps makes the frozen perturbation track the exact Borda jump.
SAC_EPS = 3e-7
print(f"A1 = A2 = {A1:.3e} m^2   AT = {AT:.3e} m^2   A1/AT = {A1/AT:.1f}   beta_min = {AT/A2:.4f}")

A1 = A2 = 1.425e-03 m^2   AT = 3.421e-05 m^2   A1/AT = 41.7   beta_min = 0.0240


## 1. Mean flow — upstream pressure rise (De Domenico Fig. 5 / 9)

For the **orifice plate** ($\beta=\beta_{\min}$) the flow contracts isentropically to
the throat and then expands as a jet with a momentum-conserving (lossy) jump.  We
sweep the inlet total pressure, solve the steady mean flow, and read the throat Mach
and the upstream/downstream static-pressure ratio $p_1/p_2$.  The loss shows up as
$p_1/p_2 > 1$ (upstream static pressure is higher), growing with Mach — exactly the
orifice signature in their Fig. 5.  The fully isentropic nozzle would recover all of
it ($p_1/p_2 = 1$ for $A_1=A_2$).

In [2]:
def solve_orifice(pt_in):
    # Cambridge orifice plate: isentropic contraction A1->AT + Borda expansion AT->A2
    net = [cat.total_pressure_inlet(pt_in, 300.0), cat.isentropic_area_change(),
           cat.sudden_area_change(eps=SAC_EPS), cat.pressure_outlet(101325.0, 300.0)]
    prob = cat.build_problem(CFG, net, [(0, 1, A1), (1, 2, AT), (2, 3, A2)], 1.0, 101325.0, CP * 300.0)
    res = solve(prob)
    return prob, res, states_table(prob, res.x)

pts = np.linspace(103000.0, 182000.0, 30)
MT, p_ratio = [], []
for pt in pts:
    _, res, est = solve_orifice(pt)
    if not res.converged or est[ES_M, 1] >= 1.0:
        continue
    MT.append(est[ES_M, 1]); p_ratio.append(est[ES_P, 0] / est[ES_P, 2])
MT, p_ratio = np.array(MT), np.array(p_ratio)

fig = go.Figure()
fig.add_trace(go.Scatter(x=MT, y=p_ratio, mode="lines+markers", name="orifice (FNS)", line_color=COLORWAY[0]))
fig.add_hline(y=1.0, line=dict(dash="dash", color=COLORWAY[4]),
              annotation_text="isentropic nozzle (no loss)", annotation_position="bottom right")
fig.update_layout(title="Upstream static-pressure rise across the orifice (De Domenico Fig. 5)",
                  xaxis_title="throat Mach  M_T", yaxis_title="p_1 / p_2", height=440, width=720)
fig.show()
print(f"throat Mach span: {MT.min():.2f} .. {MT.max():.2f}   (subsonic; M_T = 1 is the choke / v1 limit)")

throat Mach span: 0.16 .. 0.98   (subsonic; M_T = 1 is the choke / v1 limit)


## 2. Analytical anchor — compact scattering vs an independent jump composition

The De Domenico model solves a 9-unknown linear system $\mathbf{M}\,\mathbf{X} =
\mathbf{a}_1 P^+_1 + \mathbf{a}_2 P^-_2 + \mathbf{a}_3 \sigma_1$ (their Eq. 19): a
**scattering matrix** in the $(P^+,P^-,\sigma)$ flavour.  We reproduce it
*independently* by composing the analytic compact jumps (mass + total enthalpy +
[$p_t$ | momentum]), linearised by complex step — sharing no code with the FNS
kernels — and compare to the assembled FNS compact matrix at a representative
operating point.

In [3]:
# --- independent analytic compact jumps (primitive vars), complex-step safe ---
def _derived(y):
    rho, u, p = y
    ht = K * p / rho + 0.5 * u * u
    M = u / (GAMMA * p / rho) ** 0.5
    return ht, p * (1.0 + 0.5 * (GAMMA - 1.0) * M * M) ** (GAMMA / (GAMMA - 1.0))

def r_isen(y0, y1, A0, A1):
    rho0, u0, p0 = y0; rho1, u1, p1 = y1
    ht0, pt0 = _derived(y0); ht1, pt1 = _derived(y1)
    return np.array([rho1*u1*A1 - rho0*u0*A0, ht1 - ht0, pt0 - pt1])

def r_borda(y0, y1, A0, A1):              # De Domenico Eq. 5/6 (small-port pressure on the step)
    rho0, u0, p0 = y0; rho1, u1, p1 = y1
    ht0, _ = _derived(y0); ht1, _ = _derived(y1)
    md0, md1 = rho0*u0*A0, rho1*u1*A1
    ps = p0 if A0 <= A1 else p1
    return np.array([md1 - md0, ht1 - ht0, (md1*u1 + p1*A1) - (md0*u0 + p0*A0) - ps*(A1 - A0)])

def jump_tm(rf, y0, y1, *a):
    y0 = np.asarray(y0, complex); y1 = np.asarray(y1, complex); h = 1e-30
    Ja = np.zeros((3, 3)); Jb = np.zeros((3, 3))
    for k in range(3):
        yp = y0.copy(); yp[k] += 1j*h; Ja[:, k] = rf(yp, y1, *a).imag/h
        yp = y1.copy(); yp[k] += 1j*h; Jb[:, k] = rf(y0, yp, *a).imag/h
    return -np.linalg.solve(Jb, Ja)

prim = lambda est, e: np.array([est[ES_RHO, e], est[ES_U, e], est[ES_P, e]])

prob, res, est = solve_orifice(140000.0)               # M_T ~ 0.71
T_char = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).transfer_matrix(0, 2)[0]
R0 = char_to_dq(est[ES_RHO, 0], est[ES_C, 0]); R2 = char_to_dq(est[ES_RHO, 2], est[ES_C, 2])
T_fns_prim = (R2 @ T_char @ np.linalg.inv(R0)).real
T_ref_prim = (jump_tm(r_borda, prim(est, 1), prim(est, 2), AT, A2)
              @ jump_tm(r_isen, prim(est, 0), prim(est, 1), A1, AT)).real

rel_err = np.max(np.abs(T_fns_prim - T_ref_prim)) / np.max(np.abs(T_ref_prim))
print(f"orifice at M_T = {est[ES_M,1]:.3f},  p_t loss = {est[ES_PT,1]-est[ES_PT,2]:.0f} Pa")
print(f"max relative |FNS compact - independent composition| (primitive TM) = {rel_err:.2e}")

orifice at M_T = 0.714,  p_t loss = 38653 Pa
max relative |FNS compact - independent composition| (primitive TM) = 1.50e-08


In [4]:
# Read off the De Domenico coefficients from the FNS scattering matrix in the
# (P+, P-, sigma) = `riemann` flavour.  Incoming [P+_1, sigma_1, P-_2], outgoing
# [P-_1, P+_2, sigma_2]; T+/T- carry De Domenico's gamma*p ratio (Eq. 16).
S = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0].real
p1, p2 = est[ES_P, 0], est[ES_P, 2]
coeffs = {
    "R+ (acoustic refl. up)":   S[0, 0],
    "R- (acoustic refl. down)": S[1, 2],
    "T+ (transmission down)":   S[1, 0] * (p2 / p1),
    "T- (transmission up)":     S[0, 2] * (p1 / p2),
    "entropy -> upstream sound":   S[0, 1],
    "entropy -> downstream sound": S[1, 1],
}
print(f"De Domenico transfer functions at M_T = {est[ES_M,1]:.3f} (orifice):")
for k, v in coeffs.items():
    print(f"  {k:30s} = {v:+.4f}")

De Domenico transfer functions at M_T = 0.714 (orifice):
  R+ (acoustic refl. up)         = +0.9705
  R- (acoustic refl. down)       = +0.9192
  T+ (transmission down)         = +0.0632
  T- (transmission up)           = +0.0339
  entropy -> upstream sound      = -0.0062
  entropy -> downstream sound    = +0.0082


## 3. Transfer functions vs throat Mach (De Domenico Fig. 6)

Sweeping the operating point gives the compact acoustic ($R^\pm,T^\pm$) and entropic
transfer functions of the orifice plate versus throat Mach.  The **entropy→sound**
coefficients (the *indirect noise*) grow with Mach — the conversion an acoustics-only
model cannot capture — and the upstream acoustic reflection $R^+$ changes sign at low
Mach, the orifice cancellation discussed in their §3.

In [5]:
def dedomenico_coeffs(pt_in):
    prob, res, est = solve_orifice(pt_in)
    if not res.converged or est[ES_M, 1] >= 1.0:
        return None
    S = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0].real
    p1, p2 = est[ES_P, 0], est[ES_P, 2]
    return est[ES_M, 1], dict(Rp=S[0, 0], Rm=S[1, 2], Tp=S[1, 0]*(p2/p1), Tm=S[0, 2]*(p1/p2),
                              SR=S[0, 1], ST=S[1, 1])

rows = [r for r in (dedomenico_coeffs(pt) for pt in np.linspace(103000.0, 182000.0, 40)) if r]
m = np.array([r[0] for r in rows])
C = {k: np.array([r[1][k] for r in rows]) for k in rows[0][1]}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Acoustic transfer functions", "Entropy -> sound (indirect noise)"),
                    horizontal_spacing=0.12)
for k, lab, c in [("Rp", "R+", 0), ("Rm", "R-", 1), ("Tp", "T+", 2), ("Tm", "T-", 3)]:
    fig.add_trace(go.Scatter(x=m, y=C[k], name=lab, line_color=COLORWAY[c]), row=1, col=1)
for k, lab, c in [("SR", "entropy->up", 5), ("ST", "entropy->down", 6)]:
    fig.add_trace(go.Scatter(x=m, y=C[k], name=lab, line_color=COLORWAY[c]), row=1, col=2)
fig.add_hline(y=0.0, line=dict(dash="dot", color="#9aa5b1"), row=1, col=1)
fig.update_xaxes(title_text="throat Mach  M_T", row=1, col=1)
fig.update_xaxes(title_text="throat Mach  M_T", row=1, col=2)
fig.update_layout(title="Compact transfer functions of the Cambridge orifice plate (De Domenico Fig. 6)",
                  height=460, width=980)
fig.show()

## 4. The isentropic-nozzle limit ($\beta = 1$)

De Domenico note that, in the compact approximation, a **fully isentropic nozzle with
$A_1=A_2$** has acoustic transmission $T^+ = 1$ and reflection $R^+ = 0$ (mass, total
enthalpy and entropy simply pass through).  At the Cambridge contraction
($A_1/A_T\approx 42$) the isentropic throat is *always choked* ($M_T=1$) — the v1
subsonic limit, and exactly the $M_T = 1$ discontinuity in their Fig. 6 — so we
confirm the limit on a milder, genuinely-subsonic geometry.

In [6]:
# moderate isentropic nozzle: contraction A1->AT then isentropic expansion AT->A2 (= A1)
a1 = a2 = 0.020; at = 0.010
net = [cat.mass_flow_inlet(2.0, 300.0), cat.isentropic_area_change(),
       cat.isentropic_area_change(), cat.pressure_outlet(101325.0, 300.0)]
prob = cat.build_problem(CFG, net, [(0, 1, a1), (1, 2, at), (2, 3, a2)], 3.0, 101325.0, CP * 300.0)
res = solve(prob); est = states_table(prob, res.x)
S = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0]
print(f"isentropic nozzle, A1 = A2, M_T = {est[ES_M,1]:.3f},  p_t loss = {est[ES_PT,0]-est[ES_PT,2]:.2e} Pa")
print(f"  R+ = {S[0,0].real:+.6f}   (De Domenico: 0)")
print(f"  T+ = {(S[1,0]*(est[ES_P,2]/est[ES_P,0])).real:+.6f}   (De Domenico: 1)")

isentropic nozzle, A1 = A2, M_T = 0.566,  p_t loss = 8.79e-06 Pa
  R+ = -0.000000   (De Domenico: 0)
  T+ = +1.000000   (De Domenico: 1)


## Summary

* The De Domenico lossy-nozzle cases are built from **`isentropic_area_change` +
  `sudden_area_change`** — no new elements; the sudden-jump momentum law is their
  Eq. 5/6.
* The assembled **compact scattering matrix** (in the `riemann` = $(P^+,P^-,\sigma)$
  flavour) matches an **independent** composition of the analytic jumps to solver
  accuracy, and reproduces the orifice mean-flow loss (Fig. 5) and the acoustic +
  entropic transfer functions (Fig. 6).
* Scope is **subsonic throats** (v1); the isentropic-nozzle choke at $M_T=1$ is the
  De Domenico discontinuity, deferred here.
* The `sudden_area_change` smooth switch biases the *raw* transfer matrix by
  $O(\varepsilon)$; the per-element `eps` (used above) sharpens it.  The physical
  scattering coefficients are $\varepsilon$-robust regardless (see
  `tests/test_perturbation_dedomenico.py`).
